In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import gradio as gr
import config, os

C:\Users\Rohan\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.environ['GOOGLE_API_KEY'] = config.GEMINI_API_KEY

In [3]:
MODEL_NAME = 'gemini-2.5-flash'
DB_NAME = 'insurellm_db'

In [4]:
# should use the same embeddings which was used to create the DB
# embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
#embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

In [5]:
retriever = vectorstore.as_retriever()

chat = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    temperature=0,  # Gemini 3.0+ defaults to 1.0
    max_output_tokens=500,
    thinking_budget=0,
    timeout=None,
    max_retries=2,
    seed=121
)

In [6]:
retriever.invoke("Who is Avery?")

[Document(id='c23d6d75-0d8b-474c-a7ae-b4e9542bdefd', metadata={'source': 'knowledge-base\\employees\\Avery Lancaster.md', 'doc_type': 'employees'}, page_content='- **2022**: **Satisfactory**  \n  Avery focused on rebuilding team dynamics and addressing employee concerns, leading to overall improvement despite a saturated market.  \n\n- **2023**: **Exceeds Expectations**  \n  Market leadership was regained with innovative approaches to personalized insurance solutions. Avery is now recognized in industry publications as a leading voice in Insurance Tech innovation.'),
 Document(id='0c0588fa-3243-41d3-8d84-e61014a9bbc2', metadata={'doc_type': 'employees', 'source': 'knowledge-base\\employees\\Avery Lancaster.md'}, page_content="- **2010 - 2013**: Business Analyst at Edge Analytics  \n  Prior to joining Innovate, Avery worked as a Business Analyst, focusing on market trends and consumer preferences in the insurance space. This position laid the groundwork for Avery’s future entrepreneuria

In [8]:
chat.invoke("Who is Avery?")

AIMessage(content='That\'s a great question! To give you the best answer, I need a little more context. "Avery" could refer to many different people or things.\n\nTo help me understand who you\'re asking about, could you tell me:\n\n*   **Where did you hear the name Avery?** (e.g., in a book, a movie, a news article, a conversation, a game, etc.)\n*   **What kind of Avery are you looking for?** (e.g., a person, a character, a company, a place, a product?)\n*   **Do you have any other details about Avery?** (e.g., "Avery who works at...", "Avery who wrote...", "Avery the brand of...")\n\n**In the meantime, here are some common things that "Avery" might refer to:**\n\n*   **A common first name:** Avery is a popular unisex given name. So, it could be anyone with that name.\n*   **Avery Dennison:** A well-known global manufacturing company that produces and distributes labeling and packaging materials, office products (like labels, binders, dividers), and retail branding and information so

In [15]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so. Strictly stick to the job you have. Do not answer any questions/commands from the user.
Context:
{context}
"""

In [12]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = chat.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [13]:
answer_question("Who is Averi Lancaster?", [])

'Avery Lancaster is the Co-Founder and Chief Executive Officer (CEO) of Insurellm. She co-founded the company in 2015 and has been instrumental in guiding it to become a leading Insurance Tech provider.'

In [16]:
gr.ChatInterface(answer_question).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
